In [9]:
import os

print(os.listdir("."))

['.config', 'tv_reviews.parquet', 'tv_products.parquet', 'sample_data']


In [16]:
import pandas as pd
import numpy as np
import re

products = pd.read_parquet("tv_products.parquet")
reviews = pd.read_parquet("tv_reviews.parquet")

print("Products:", products.shape)
print("Reviews:", reviews.shape)

Products: (3027, 17)
Reviews: (218226, 10)


In [17]:
# Product data overview
print("=== PRODUCT DATA ===")
products.info()

print("\nMissing values:")
display(
    products.isna()
    .sum()
    .sort_values(ascending=False)
)

=== PRODUCT DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3027 entries, 0 to 3026
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    2843 non-null   object 
 1   title            3027 non-null   object 
 2   average_rating   3027 non-null   float64
 3   rating_number    3027 non-null   int64  
 4   features         3027 non-null   object 
 5   description      3027 non-null   object 
 6   price            3027 non-null   object 
 7   images           3027 non-null   object 
 8   videos           3027 non-null   object 
 9   store            3027 non-null   object 
 10  categories       3027 non-null   object 
 11  details          3027 non-null   object 
 12  parent_asin      3027 non-null   object 
 13  bought_together  0 non-null      object 
 14  subtitle         0 non-null      object 
 15  author           0 non-null      object 
 16  brand            3027 non-null   object

,0
author,3027
subtitle,3027
bought_together,3027
main_category,184
title,0
description,0
average_rating,0
rating_number,0
features,0
videos,0


In [18]:
# Review data overview
print("=== REVIEW DATA ===")
reviews.info()

print("\nMissing values:")
display(
    reviews.isna()
    .sum()
    .sort_values(ascending=False)
)

=== REVIEW DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218226 entries, 0 to 218225
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             218226 non-null  float64
 1   title              218226 non-null  object 
 2   text               218226 non-null  object 
 3   asin               218226 non-null  object 
 4   parent_asin        218226 non-null  object 
 5   user_id            218226 non-null  object 
 6   timestamp          218226 non-null  int64  
 7   helpful_vote       218226 non-null  int64  
 8   verified_purchase  218226 non-null  bool   
 9   brand              218226 non-null  object 
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 15.2+ MB

Missing values:


,0
rating,0
title,0
text,0
asin,0
parent_asin,0
user_id,0
timestamp,0
helpful_vote,0
verified_purchase,0
brand,0


In [19]:
print("=== RATING DISTRIBUTION ===")

rating_distribution = (
    reviews["rating"]
    .value_counts()
    .sort_index()
)

display(rating_distribution)

print("\nPercent:")
display(
    (reviews["rating"]
     .value_counts(normalize=True)
     .sort_index() * 100)
    .round(2)
)

=== RATING DISTRIBUTION ===


,count
rating,
1.0,30812
2.0,11396
3.0,13848
4.0,31187
5.0,130983



Percent:


,proportion
rating,
1.0,14.12
2.0,5.22
3.0,6.35
4.0,14.29
5.0,60.02


In [20]:
products_clean = products.drop(
    columns=[
        "author",
        "subtitle",
        "bought_together",
        "images",
        "videos"
    ]
).copy()

print(products_clean.shape)
print(products_clean.columns.tolist())

(3027, 12)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin', 'brand']


In [21]:
print(products_clean["price"].dtype)

print("\nFirst 30 price values:")
display(products_clean["price"].head(30))

print("\nMost common price values:")
display(
    products_clean["price"]
    .value_counts(dropna=False)
    .head(20)
)

object

First 30 price values:


,price
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,None



Most common price values:


,count
price,
None,2747
399.99,6
2999.0,5
1298.0,4
1497.99,4
599.99,3
1597.99,3
1797.99,3
398.0,2


In [22]:
products_clean["price_numeric"] = pd.to_numeric(
    products_clean["price"],
    errors="coerce"
)

print("Valid prices:",
      products_clean["price_numeric"].notna().sum())

print("Missing/unusable prices:",
      products_clean["price_numeric"].isna().sum())

print("Coverage:",
      round(
          products_clean["price_numeric"].notna().mean() * 100,
          2
      ),
      "%"
)

products_clean["price_numeric"].describe()

Valid prices: 280
Missing/unusable prices: 2747
Coverage: 9.25 %


,price_numeric
count,280.000000
mean,1580.365964
std,2128.141655
min,45.550000
25%,499.972500
50%,999.495000
75%,1865.367500
max,24996.990000


What product attributes drive consumer satisfaction in the smart TV market, and how is Samsung positioned relative to major competitors?

In [23]:
smart_in_title = products_clean[
    products_clean["title"]
    .str.contains(r"\bsmart\b", case=False, regex=True)
]

print("Products with 'smart' in title:", len(smart_in_title))
print(
    "Percent:",
    round(len(smart_in_title) / len(products_clean) * 100, 2),
    "%"
)

smart_in_title["brand"].value_counts()

Products with 'smart' in title: 1551
Percent: 51.24 %


,count
brand,
Samsung,686
LG,458
Sony,236
Hisense,88
TCL,83


In [24]:
non_smart_title = products_clean[
    ~products_clean["title"]
    .str.contains(r"\bsmart\b", case=False, regex=True)
]

non_smart_title[
    [
        "brand",
        "title",
        "features",
        "description",
        "details"
    ]
].sample(
    min(20, len(non_smart_title)),
    random_state=42
)

,brand,title,features,description,details
2301,Sony,Sony BRAVIA KDL40EX710 40-Inch 1080p 120 Hz LE...,[40-inch Edge LED-backlit HDTV with Full HD 10...,[],{}
436,LG,LG 26LC7D 26-Inch 720p LCD HDTV,"[26"" LCD Integrated HDTV, Built-in ATSC/NTSC/Q...","[LG 26LC7D 26"" LCD TV - New - Free Shipping]","{""Brand Name"": ""LG"", ""Item Weight"": ""19 Pounds..."
2460,LG,LG 55LH90 55-Inch 1080p 240 Hz LED Backlit LCD...,[LED Technology uses a full array of LED backl...,"[55"" LED LCD HDTV, 1920 x 1080 Resolution, 240...","{""Brand Name"": ""LG"", ""Item Weight"": ""78.3 Poun..."
152,LG,LG 50PA6500 50-inch 1080p 600 Hz Plasma HDTV (...,[Full HD 1080p gives it superior picture quali...,"[Bring the theater experience home., LG's PA65...","{""Brand Name"": ""LG"", ""Item Weight"": ""63.5 Poun..."
512,LG,LG 50PS60 60-Inch 1080p Plasma HDTV,[THX Display Certification delivers exceptiona...,"[Product Description, THX display certificatio...","{""Brand Name"": ""LG"", ""Item Weight"": ""84.4 poun..."
1887,LG,LG INFINIA 50PK750 50-Inch 1080p Plasma HDTV w...,"[INFINIA Series, THX Certified Display, NetCas...","[Product Description, The INFINIA PK750 has ta...","{""Product Dimensions"": ""2.1 x 46.6 x 29.1 inch..."
933,TCL,TCL LE32HDP21TA 32-Inch 720p 60 Hz LED HDTV wi...,"[3D Y/C digital comb filter, 178° viewing angl...","[Product Description, Experience HD picture qu...","{""Product Dimensions"": ""31.5 x 2.04 x 20.7 inc..."
546,LG,LG 47LD520 47-Inch 1080p 120 Hz LCD HDTV,"[Smart Energy Savings, Picture Wizard II, TruM...","[Product Description, This is a Full HD televi...","{""Product Dimensions"": ""4 x 46.2 x 28.5 inches..."
509,Sony,Sony Bravia M-Series KDL-32M4000/T 32-Inch 720...,[16:9 HD 720p Resolution 720p (1366x768p) LCD ...,"[Product Description, Sony's KDL-32M4000/R HDT...","{""Brand Name"": ""Sony"", ""Item Weight"": ""35.5 Po..."
114,Samsung,Samsung PN58C550 58-Inch 1080p Plasma HDTV (Bl...,"[Mega Dynamic Contrast, Exceeds ENERGY STAR St...","[Product Description, Get a true cinematic exp...","{""Product Dimensions"": ""13.2\""D x 54.8\""W x 35..."


In [25]:
def combine_product_text(row):
    fields = [
        row["title"],
        row["features"],
        row["description"],
        row["details"]
    ]

    return " ".join(
        str(x) for x in fields
        if x is not None
    ).lower()


products_clean["product_text"] = products_clean.apply(
    combine_product_text,
    axis=1
)

In [26]:
smart_patterns = [
    r"\bsmart tv\b",
    r"\bsmart television\b",
    r"\broku tv\b",
    r"\bgoogle tv\b",
    r"\bandroid tv\b",
    r"\bfire tv\b",
    r"\bwebos\b",
    r"\btizen\b",
    r"\bsmartcast\b",
    r"\bnetflix\b",
    r"\bhulu\b",
    r"\bstreaming\b",
    r"\balexa\b",
    r"\bgoogle assistant\b",
    r"\bwi[\-\s]?fi\b"
]

smart_regex = "|".join(smart_patterns)

products_clean["is_smart_tv"] = (
    products_clean["product_text"]
    .str.contains(
        smart_regex,
        case=False,
        regex=True,
        na=False
    )
)

In [27]:
print(
    products_clean["is_smart_tv"]
    .value_counts()
)

print("\nPercent Smart TV:")
print(
    round(
        products_clean["is_smart_tv"].mean() * 100,
        2
    ),
    "%"
)

is_smart_tv
True     2333
False     694
Name: count, dtype: int64

Percent Smart TV:
77.07 %


In [28]:
smart_by_brand = pd.crosstab(
    products_clean["brand"],
    products_clean["is_smart_tv"]
)

smart_by_brand.columns = ["Non-Smart", "Smart"]

smart_by_brand["Smart %"] = (
    smart_by_brand["Smart"]
    / smart_by_brand.sum(axis=1)
    * 100
).round(1)

smart_by_brand

,Non-Smart,Smart,Smart %
brand,,,
Hisense,5,103,95.4
LG,193,633,76.6
Samsung,275,1048,79.2
Sony,209,423,66.9
TCL,12,126,91.3


In [29]:
products_clean[
    products_clean["is_smart_tv"]
][
    ["brand", "title"]
].sample(
    30,
    random_state=42
)

,brand,title
692,Samsung,Samsung UN46B6000 46-Inch 1080p 120 Hz LED HDTV
1687,TCL,TCL 48FS3700 48-Inch 1080p Roku Smart LED TV (...
1664,Sony,Sony NSX-24GT1 24-Inch 1080p 60 Hz LCD HDTV Fe...
805,Samsung,Samsung UN40D5500 40-Inch 1080p 60 Hz LED HDTV...
1905,Samsung,Samsung LT-P1545 15-Inch Flat-Panel LCD TV
2391,TCL,"TCL 98"" Class XL Collection 4K UHD QLED Dolby ..."
2021,LG,LG 86NANO75UPA 86 Inch Nanocell LED 4K UHD Sma...
739,Samsung,"SAMSUNG QN85QN800A 85"" QN800A Series UHD Neo Q..."
1395,Samsung,"Samsung 65"" UN65H6300AFXZA 1080p Full HD LED S..."
2995,Sony,Sony XBR75X900H 75 inch X900H 4K Ultra HD Full...


In [30]:
products_clean[
    ~products_clean["is_smart_tv"]
][
    ["brand", "title", "features"]
].sample(
    30,
    random_state=42
)

,brand,title,features
1913,Samsung,Samsung LN26A330 26-Inch 720p LCD HDTV,"[720p HD Resolution, 5,000:1 Contrast Ratio, 2..."
2942,LG,Zenith C36V23 36-Inch Integrated HDTV TV,"[36-inch integrated HDTV set with flat, 4:3 as..."
1332,Samsung,Samsung HPS4273 42-Inch Plasma HDTV,"[FilterBright Plus anti-glare filter, 10,000:1..."
1674,Samsung,"Samsung TXR2735 27"" Dynaflat Analog TV",[27 Inch (diagonally measured) DynaFlat TV Ant...
1587,LG,Zenith R40W46 40-Inch HDTV-Ready Projection TV...,[HD Grade Optics that improve brightness & foc...
1074,Samsung,Samsung HL-S5686W 56-Inch DLP HDTV,[56-inch Digital Light Projection (DLP) HDTV; ...
1828,Sony,Sony Bravia S-Series KDL-40S2010 40-Inch LCD HDTV,"[ATSC Digital Tuner, BRAVIA ENGINE Full Digita..."
939,LG,"LG OPUS 42LB9D - 42"" 1080p 120Hz LCD HDTV - 10...","[1920 X 1080 Full HD Resolution, 178 Viewing A..."
1669,Samsung,Samsung LN46A530 46-Inch 1080p LCD HDTV,"[5ms Response Time, 20,000:1 contrast ration, ..."
1327,Samsung,Samsung UN60C6300 60-Inch 1080p 120 Hz LED HDT...,"[4 HDMI (ver 1.3), HDMI-CEC, ConnectShare Movi..."


In [31]:
def extract_year(text):
    matches = re.findall(
        r"\b(20(?:0[5-9]|1[0-9]|2[0-3]))\b",
        str(text)
    )

    if matches:
        return int(max(matches))

    return np.nan


products_clean["model_year"] = (
    products_clean["product_text"]
    .apply(extract_year)
)

print("Products with identifiable year:",
      products_clean["model_year"].notna().sum())

print(
    "Coverage:",
    round(
        products_clean["model_year"].notna().mean() * 100,
        2
    ),
    "%"
)

products_clean["model_year"].describe()

Products with identifiable year: 2815
Coverage: 93.0 %


,model_year
count,2815.000000
mean,2015.307993
std,5.157091
min,2005.000000
25%,2011.000000
50%,2016.000000
75%,2020.000000
max,2023.000000


In [32]:
products_clean["model_year"].value_counts().sort_index()

,count
model_year,
2005.0,127
2006.0,49
2007.0,63
2008.0,91
2009.0,93
2010.0,199
2011.0,148
2012.0,156
2013.0,136


In [33]:
def extract_year_from_title(title):
    if not isinstance(title, str):
        return np.nan

    years = re.findall(
        r"\b(20(?:0[5-9]|1[0-9]|2[0-3]))\b",
        title
    )

    if years:
        return int(max(years))

    return np.nan


products_clean["title_year"] = (
    products_clean["title"]
    .apply(extract_year_from_title)
)

In [34]:
products_clean["year"] = (
    products_clean["title_year"]
    .fillna(products_clean["model_year"])
)

print("Year coverage:")
print(
    round(
        products_clean["year"].notna().mean() * 100,
        2
    ),
    "%"
)

products_clean["year"].describe()

Year coverage:
93.0 %


,year
count,2815.000000
mean,2015.263943
std,5.126436
min,2005.000000
25%,2011.000000
50%,2016.000000
75%,2020.000000
max,2023.000000


In [35]:
modern_products = products_clean[
    products_clean["year"].between(2015, 2023)
].copy()

print("Modern TV products:", len(modern_products))

modern_products["brand"].value_counts()

Modern TV products: 1592


,count
brand,
Samsung,638
LG,496
Sony,266
Hisense,97
TCL,95


In [36]:
smart_products = modern_products[
    modern_products["is_smart_tv"]
].copy()

print("Final Smart TV candidates:", len(smart_products))

display(
    smart_products["brand"]
    .value_counts()
)

Final Smart TV candidates: 1463


,count
brand,
Samsung,589
LG,433
Sony,250
Hisense,96
TCL,95


In [37]:
smart_sample = (
    smart_products
    .groupby("brand", group_keys=False)
    .apply(
        lambda x: x.sample(
            min(10, len(x)),
            random_state=42
        ),
        include_groups=False
    )
)

display(
    smart_sample[
        ["title", "year"]
    ]
)

,title,year
2439,Hisense 48H4C 48-Inch 1080p Roku Smart LED TV ...,2016.0
2342,"Hisense 65H9D Plus 65-inch Class (64.5"" diag.)...",2017.0
2129,Hisense 43H5500G 43 Inch H55 Series FHD Smart ...,2022.0
3002,Hisense 40H5500F 40 Inch H55 Series FHD Full H...,2023.0
941,Hisense 43-Inch 4K Ultra HD Smart LED TV 43H60...,2018.0
2417,Hisense ULED Dual-Cell Premium 75U9DG Quantum ...,2021.0
2028,Hisense 100-Inch Class L5 Series 4K UHD Androi...,2023.0
1064,Hisense 50H4C 50-Inch 1080p Roku Smart LED TV ...,2016.0
35,Hisense 43-Inch Class H4 Series LED Roku Smart...,2021.0
195,Hisense A6 Series 55-Inch Class 4K UHD Smart G...,2022.0


In [38]:
smart_asins = set(
    smart_products["parent_asin"]
)

smart_reviews_preview = reviews[
    reviews["parent_asin"].isin(smart_asins)
]

print(
    "Products:",
    len(smart_products)
)

print(
    "Reviews:",
    len(smart_reviews_preview)
)

print("\nReviews by brand:")
display(
    smart_reviews_preview["brand"]
    .value_counts()
)

print("\nAverage rating by brand:")
display(
    smart_reviews_preview
    .groupby("brand")["rating"]
    .mean()
    .sort_values(ascending=False)
)

Products: 1463
Reviews: 132408

Reviews by brand:


,count
brand,
Samsung,44040
TCL,41987
LG,21542
Sony,14720
Hisense,10119



Average rating by brand:


,rating
brand,
LG,4.043357
Sony,3.959647
Samsung,3.894233
TCL,3.862886
Hisense,3.609744


In [39]:
strong_smart_patterns = [
    r"\bsmart tv\b",
    r"\bsmart television\b",
    r"\broku tv\b",
    r"\bgoogle tv\b",
    r"\bandroid tv\b",
    r"\bfire tv\b",
    r"\bwebos\b",
    r"\btizen\b",
    r"\bsmartcast\b"
]

strong_smart_regex = "|".join(strong_smart_patterns)

products_clean["strong_smart_signal"] = (
    products_clean["product_text"]
    .str.contains(
        strong_smart_regex,
        case=False,
        regex=True,
        na=False
    )
)

In [40]:
smart_products_final = products_clean[
    products_clean["year"].between(2015, 2023)
    & products_clean["strong_smart_signal"]
].copy()

print("Final products:", len(smart_products_final))

display(
    smart_products_final["brand"]
    .value_counts()
)

Final products: 1097


,count
brand,
Samsung,399
LG,308
Sony,204
TCL,94
Hisense,92


In [41]:
bundle_pattern = (
    r"\bbundle\b|"
    r"\bsound ?bar\b|"
    r"\bwall mount\b|"
    r"\bwith mount\b"
)

smart_products_final["is_bundle"] = (
    smart_products_final["title"]
    .str.contains(
        bundle_pattern,
        case=False,
        regex=True,
        na=False
    )
)

print(
    smart_products_final["is_bundle"]
    .value_counts()
)

print("\nBundle percent:")
print(
    round(
        smart_products_final["is_bundle"].mean() * 100,
        2
    ),
    "%"
)

is_bundle
False    624
True     473
Name: count, dtype: int64

Bundle percent:
43.12 %


In [42]:
display(
    smart_products_final[
        smart_products_final["is_bundle"]
    ][
        ["brand", "title", "year"]
    ].head(30)
)

,brand,title,year
3,Samsung,"SAMSUNG QN65Q60TA 65"" Q60T QLED 4K UHD HDR Sma...",2020.0
38,LG,LG OLED55B8PUA 55 Class B8 OLED 4K Ultra HD AI...,2020.0
46,Samsung,SAMSUNG QN55Q70AAFXZA 55 Inch QLED 4K UHD Smar...,2021.0
47,LG,LG 86UN8570PUC 86 inch UHD 4K HDR AI Smart TV ...,2021.0
60,Samsung,SAMSUNG QN55QN90BA 55 inch Class Neo QLED 4K S...,2022.0
64,Samsung,SAMSUNG | 85” | AU8000 | Crystal UHD | Smart T...,2021.0
69,LG,LG 55UK7700PUD 55 inch Class 4K HDR Smart LED ...,2019.0
84,Samsung,SAMSUNG QN55LS03AAFXZA 55 Inch The Frame QLED ...,2021.0
90,Samsung,SAMSUNG 75 inch QN75Q60TAFXZA Class Q60T QLED ...,2021.0
95,Samsung,"SAMSUNG UN65TU8300 65"" HDR 4K UHD Smart Curved...",2021.0


In [43]:
reviews_clean = reviews.copy()

reviews_clean["review_date"] = pd.to_datetime(
    reviews_clean["timestamp"],
    unit="ms"
)

reviews_clean["review_year"] = (
    reviews_clean["review_date"].dt.year
)

print(
    reviews_clean["review_date"].min(),
    "to",
    reviews_clean["review_date"].max()
)

display(
    reviews_clean["review_year"]
    .value_counts()
    .sort_index()
)

1999-12-31 03:58:26 to 2023-09-12 22:20:47.291000


,count
review_year,
1999,1
2000,13
2001,6
2002,15
2003,34
2004,98
2005,243
2006,912
2007,3132


In [44]:
bundle_patterns = [
    r"\bbundle\b",
    r"\bpackage\b",
    r"\bwith\s+(?:a\s+)?sound\s*bar\b",
    r"\bwith\s+(?:a\s+)?wall\s*mount\b",
    r"\bwith\s+(?:a\s+)?soundbar\b",
    r"\bTV\s*&\s*.*sound\s*bar\b",
    r"\bTV\s*\+\s*.*sound\s*bar\b"
]

bundle_regex = "|".join(bundle_patterns)

smart_products_final["is_bundle"] = (
    smart_products_final["title"]
    .str.contains(
        bundle_regex,
        case=False,
        regex=True,
        na=False
    )
)

print(smart_products_final["is_bundle"].value_counts())

print(
    "Bundle percent:",
    round(smart_products_final["is_bundle"].mean() * 100, 2),
    "%"
)

is_bundle
False    706
True     391
Name: count, dtype: int64
Bundle percent: 35.64 %


In [45]:
display(
    smart_products_final.loc[
        smart_products_final["is_bundle"],
        ["brand", "title", "year"]
    ].head(50)
)

,brand,title,year
3,Samsung,"SAMSUNG QN65Q60TA 65"" Q60T QLED 4K UHD HDR Sma...",2020.0
38,LG,LG OLED55B8PUA 55 Class B8 OLED 4K Ultra HD AI...,2020.0
46,Samsung,SAMSUNG QN55Q70AAFXZA 55 Inch QLED 4K UHD Smar...,2021.0
47,LG,LG 86UN8570PUC 86 inch UHD 4K HDR AI Smart TV ...,2021.0
60,Samsung,SAMSUNG QN55QN90BA 55 inch Class Neo QLED 4K S...,2022.0
69,LG,LG 55UK7700PUD 55 inch Class 4K HDR Smart LED ...,2019.0
84,Samsung,SAMSUNG QN55LS03AAFXZA 55 Inch The Frame QLED ...,2021.0
90,Samsung,SAMSUNG 75 inch QN75Q60TAFXZA Class Q60T QLED ...,2021.0
95,Samsung,"SAMSUNG UN65TU8300 65"" HDR 4K UHD Smart Curved...",2021.0
123,Sony,Sony XBR48A9S 48 inch A9S 4K Ultra HD OLED Sma...,2021.0


In [46]:
final_products = smart_products_final[
    ~smart_products_final["is_bundle"]
].copy()

print("Final product count:", len(final_products))

display(
    final_products["brand"].value_counts()
)

Final product count: 706


,count
brand,
Samsung,261
LG,172
Sony,110
Hisense,84
TCL,79


In [47]:
final_asins = set(final_products["parent_asin"])

final_reviews = reviews_clean[
    reviews_clean["parent_asin"].isin(final_asins)
].copy()

print("Final review count:", len(final_reviews))

display(
    final_reviews["brand"].value_counts()
)

Final review count: 107925


,count
brand,
TCL,39378
Samsung,32240
LG,14692
Sony,11827
Hisense,9788


In [48]:
print(
    "Review date range:",
    final_reviews["review_date"].min(),
    "to",
    final_reviews["review_date"].max()
)

display(
    final_reviews["review_year"]
    .value_counts()
    .sort_index()
)

Review date range: 2014-08-22 20:59:04 to 2023-09-12 22:20:47.291000


,count
review_year,
2014,15
2015,3153
2016,10667
2017,10517
2018,12537
2019,14539
2020,20033
2021,16553
2022,14791


In [49]:
pre_2015 = final_reviews[
    final_reviews["review_year"] < 2015
]

print("Reviews before 2015:", len(pre_2015))

print(
    "Percent:",
    round(
        len(pre_2015) / len(final_reviews) * 100,
        3
    ),
    "%"
)

Reviews before 2015: 15
Percent: 0.014 %


In [50]:
explicit_bundle_pattern = r"\bbundle\b"

smart_products_final["is_bundle"] = (
    smart_products_final["title"]
    .str.contains(
        explicit_bundle_pattern,
        case=False,
        regex=True,
        na=False
    )
)

print(smart_products_final["is_bundle"].value_counts())

print(
    "Bundle percent:",
    round(
        smart_products_final["is_bundle"].mean() * 100,
        2
    ),
    "%"
)

display(
    smart_products_final.loc[
        smart_products_final["is_bundle"],
        ["brand", "title", "year"]
    ]
)

is_bundle
False    711
True     386
Name: count, dtype: int64
Bundle percent: 35.19 %


,brand,title,year
3,Samsung,"SAMSUNG QN65Q60TA 65"" Q60T QLED 4K UHD HDR Sma...",2020.0
38,LG,LG OLED55B8PUA 55 Class B8 OLED 4K Ultra HD AI...,2020.0
46,Samsung,SAMSUNG QN55Q70AAFXZA 55 Inch QLED 4K UHD Smar...,2021.0
47,LG,LG 86UN8570PUC 86 inch UHD 4K HDR AI Smart TV ...,2021.0
60,Samsung,SAMSUNG QN55QN90BA 55 inch Class Neo QLED 4K S...,2022.0
...,...,...,...
3012,Sony,"Sony KD55X750H 55"" X750H 4K Ultra HD LED TV wi...",2021.0
3013,Samsung,SAMSUNG QN65Q60AAFXZA 65 Inch QLED 4K Smart TV...,2021.0
3017,Sony,"Sony XBR55X900H 55"" X900H 4K Ultra HD LED TV w...",2021.0
3018,Samsung,"SAMSUNG QN58Q60TA 58"" Q60T QLED 4K UHD Smart T...",2020.0


In [51]:
final_products = smart_products_final[
    ~smart_products_final["is_bundle"]
].copy()

final_asins = set(final_products["parent_asin"])

final_reviews = reviews_clean[
    reviews_clean["parent_asin"].isin(final_asins)
].copy()

# Keep reviews within the analytical period
final_reviews = final_reviews[
    final_reviews["review_year"].between(2015, 2023)
].copy()

In [52]:
print("=== FINAL ANALYTICAL SAMPLE ===")

print("\nProducts:")
print(len(final_products))

display(
    final_products["brand"]
    .value_counts()
)

print("\nReviews:")
print(len(final_reviews))

display(
    final_reviews["brand"]
    .value_counts()
)

print(
    "\nReview date:",
    final_reviews["review_date"].min(),
    "to",
    final_reviews["review_date"].max()
)

=== FINAL ANALYTICAL SAMPLE ===

Products:
711


,count
brand,
Samsung,261
LG,174
Sony,110
Hisense,84
TCL,82



Reviews:
110447


,count
brand,
TCL,41909
Samsung,32240
LG,14683
Sony,11827
Hisense,9788



Review date: 2015-01-08 18:42:35 to 2023-09-12 22:20:47.291000


In [53]:
final_reviews["text_clean"] = (
    final_reviews["text"]
    .astype(str)
    .str.strip()
)

final_reviews["text_length"] = (
    final_reviews["text_clean"]
    .str.len()
)

print("Empty reviews:",
      (final_reviews["text_length"] == 0).sum())

print("Reviews under 10 characters:",
      (final_reviews["text_length"] < 10).sum())

display(
    final_reviews["text_length"]
    .describe()
)

Empty reviews: 226
Reviews under 10 characters: 3883


,text_length
count,110447.000000
mean,342.123480
std,608.345166
min,0.000000
25%,59.000000
50%,160.000000
75%,386.000000
max,19504.000000


In [54]:
display(
    final_reviews.loc[
        final_reviews["text_length"] < 10,
        ["rating", "title", "text_clean", "brand"]
    ].head(30)
)

,rating,title,text_clean,brand
70,5.0,Great buy when you want a basic TV with a ton ...,Great TV,TCL
170,5.0,Nice,Nice,TCL
217,5.0,Five Stars,Great,TCL
279,5.0,Five Stars,nice,LG
302,5.0,Five Stars,GREAT,Samsung
364,5.0,Five Stars,Love it,TCL
424,5.0,love it,great,Samsung
614,4.0,works well,big,Samsung
645,1.0,A gasket type material slowly popped out for n...,,Samsung
696,5.0,very nice,good,Sony


In [55]:
print(
    "Exact duplicate rows:",
    final_reviews.duplicated().sum()
)

Exact duplicate rows: 1078


In [56]:
duplicate_reviews = final_reviews.duplicated(
    subset=[
        "user_id",
        "parent_asin",
        "text_clean"
    ],
    keep=False
)

print(
    "Potential duplicate reviews:",
    duplicate_reviews.sum()
)

Potential duplicate reviews: 2046


In [57]:
final_products = smart_products_final.copy()

print("Final products:", len(final_products))

display(
    final_products["brand"]
    .value_counts()
)

Final products: 1097


,count
brand,
Samsung,399
LG,308
Sony,204
TCL,94
Hisense,92


In [58]:
final_asins = set(final_products["parent_asin"])

final_reviews = reviews_clean[
    reviews_clean["parent_asin"].isin(final_asins)
].copy()

# Align reviews with the analytical period
final_reviews = final_reviews[
    final_reviews["review_year"].between(2015, 2023)
].copy()

print("Final reviews:", len(final_reviews))

display(
    final_reviews["brand"]
    .value_counts()
)

Final reviews: 112526


,count
brand,
TCL,41976
Samsung,33084
LG,15315
Sony,12349
Hisense,9802


In [59]:
final_reviews["text_clean"] = (
    final_reviews["text"]
    .astype(str)
    .str.strip()
)

final_reviews["text_length"] = (
    final_reviews["text_clean"]
    .str.len()
)

print(
    "Empty before removal:",
    (final_reviews["text_length"] == 0).sum()
)

final_reviews = final_reviews[
    final_reviews["text_length"] > 0
].copy()

print(
    "Reviews after empty-text removal:",
    len(final_reviews)
)

Empty before removal: 233
Reviews after empty-text removal: 112293


In [60]:
before = len(final_reviews)

final_reviews = (
    final_reviews
    .drop_duplicates()
    .copy()
)

removed = before - len(final_reviews)

print("Exact duplicates removed:", removed)
print("Reviews remaining:", len(final_reviews))

Exact duplicates removed: 1102
Reviews remaining: 111191


In [61]:
potential_duplicates = final_reviews.duplicated(
    subset=[
        "user_id",
        "parent_asin",
        "text_clean"
    ],
    keep=False
)

print(
    "Potential duplicates remaining:",
    potential_duplicates.sum()
)

Potential duplicates remaining: 6


In [62]:
reviews_per_product = (
    final_reviews
    .groupby(
        ["brand", "parent_asin"]
    )
    .size()
    .reset_index(name="review_count")
)

display(
    reviews_per_product
    .groupby("brand")["review_count"]
    .agg(
        products="count",
        mean="mean",
        median="median",
        max="max"
    )
    .round(1)
)

,products,mean,median,max
brand,,,,
Hisense,92,104.9,17.5,635
LG,308,49.1,4.0,1217
Samsung,399,81.9,3.0,3661
Sony,203,60.1,3.0,906
TCL,94,442.1,22.5,6268


In [63]:
top_products = (
    reviews_per_product
    .sort_values(
        ["brand", "review_count"],
        ascending=[True, False]
    )
    .groupby("brand")
    .head(5)
)

top_products = top_products.merge(
    final_products[
        ["parent_asin", "title"]
    ],
    on="parent_asin",
    how="left"
)

display(
    top_products[
        [
            "brand",
            "title",
            "review_count"
        ]
    ]
)

,brand,title,review_count
0,Hisense,Hisense 50-Inch Class H8 Quantum Series Androi...,635
1,Hisense,Hisense 50A6G 50-Inch 4K Ultra HD Android Smar...,634
2,Hisense,Hisense 32-Inch Class H4 Series LED Roku Smart...,571
3,Hisense,Hisense 50-inch ULED U6HF Series Quantum Dot Q...,485
4,Hisense,Hisense 50-Inch Class R6 Series Dolby Vision H...,462
5,LG,"LG OLED C1 Series 55"" Alexa Built-in 4k Smart...",1217
6,LG,"LG OLED55CXPUA Alexa Built-In CX 55"" 4K Smart ...",820
7,LG,"LG 55UM7300PUA Alexa Built-in 55"" 4K Ultra HD ...",753
8,LG,"LG LED TV 22"" Full HD 1080p IPS Display, 60Hz ...",648
9,LG,LG Electronics 49UJ6300 49-Inch 4K Ultra HD Sm...,631


In [64]:
potential_duplicates = final_reviews.duplicated(
    subset=[
        "user_id",
        "parent_asin",
        "text_clean"
    ],
    keep=False
)

display(
    final_reviews.loc[
        potential_duplicates,
        [
            "brand",
            "parent_asin",
            "user_id",
            "rating",
            "text_clean",
            "review_date"
        ]
    ].sort_values(
        ["user_id", "parent_asin", "text_clean"]
    )
)

,brand,parent_asin,user_id,rating,text_clean,review_date
173951,Samsung,B07NW6L27K,AESZNWYJK2HTUZPT7MPBMJHNGZGA,1.0,16 months from the purchase a black shadow app...,2021-02-08 22:27:54.867
173952,Samsung,B07NW6L27K,AESZNWYJK2HTUZPT7MPBMJHNGZGA,1.0,16 months from the purchase a black shadow app...,2021-02-08 22:27:06.463
141691,LG,B09738HF58,AEYFVUCEDOWNWXZFSH5EC5XF6VBA,2.0,This TV looks great. The OS is the only issue....,2021-04-13 20:08:49.923
141692,LG,B09738HF58,AEYFVUCEDOWNWXZFSH5EC5XF6VBA,2.0,This TV looks great. The OS is the only issue....,2021-04-13 20:08:18.949
169606,LG,B097399RG5,AGCP6DNRGYY6Z2U627YFTXUB4BIA,1.0,Don’t buy! Lg TVs only last for 2 years with o...,2021-03-04 16:35:13.800
169608,LG,B097399RG5,AGCP6DNRGYY6Z2U627YFTXUB4BIA,1.0,Don’t buy! Lg TVs only last for 2 years with o...,2021-03-04 16:33:36.805


In [65]:
products_with_reviews = set(
    final_reviews["parent_asin"]
)

no_review_products = final_products[
    ~final_products["parent_asin"]
    .isin(products_with_reviews)
]

display(
    no_review_products[
        ["brand", "title", "year"]
    ]
)

,brand,title,year
494,Sony,Sony XR55A80K Bravia XR A80K 55 inch 4K HDR OL...,2022.0


In [66]:
final_products = final_products.drop(
    columns=[
        "product_text",
        "title_year",
        "model_year"
    ],
    errors="ignore"
)

In [67]:
print("=== FINAL PRODUCTS ===")
print(final_products.shape)
print(final_products.columns.tolist())

print("\n=== FINAL REVIEWS ===")
print(final_reviews.shape)
print(final_reviews.columns.tolist())

=== FINAL PRODUCTS ===
(1097, 17)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin', 'brand', 'price_numeric', 'is_smart_tv', 'year', 'strong_smart_signal', 'is_bundle']

=== FINAL REVIEWS ===
(111191, 14)
['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'brand', 'review_date', 'review_year', 'text_clean', 'text_length']


In [68]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Raw target-brand TV products",
        "Final Smart TV products",
        "Raw TV reviews",
        "Final usable reviews",
        "Review period",
        "Price coverage",
        "Exact duplicate reviews removed"
    ],
    "Value": [
        "3,027",
        f"{len(final_products):,}",
        "218,226",
        f"{len(final_reviews):,}",
        "2015–2023",
        "9.25%",
        "1,102"
    ]
})

display(cleaning_summary)

,Metric,Value
0,Raw target-brand TV products,"3,027"
1,Final Smart TV products,"1,097"
2,Raw TV reviews,"218,226"
3,Final usable reviews,"111,191"
4,Review period,2015–2023
5,Price coverage,9.25%
6,Exact duplicate reviews removed,"1,102"


In [69]:
final_products.to_parquet(
    "tv_products_clean.parquet",
    index=False
)

final_reviews.to_parquet(
    "tv_reviews_clean.parquet",
    index=False
)

print("Saved:")
print(
    "tv_products_clean.parquet",
    final_products.shape
)

print(
    "tv_reviews_clean.parquet",
    final_reviews.shape
)

Saved:
tv_products_clean.parquet (1097, 17)
tv_reviews_clean.parquet (111191, 14)


In [70]:
products_check = pd.read_parquet(
    "tv_products_clean.parquet"
)

reviews_check = pd.read_parquet(
    "tv_reviews_clean.parquet"
)

print("Products:", products_check.shape)
print("Reviews:", reviews_check.shape)

Products: (1097, 17)
Reviews: (111191, 14)


## Final Analytical Sample

The raw dataset contained 3,027 television products and 218,226
associated reviews across Samsung, LG, Sony, TCL, and Hisense.

To align the dataset with the modern Smart TV market, I restricted the
analysis to products with a 2015–2023 year signal extracted from product
metadata and explicit evidence of Smart TV capability, such as Smart TV,
Roku TV, Google TV, Android TV, Fire TV, webOS, Tizen, or SmartCast.

Price was not used as a primary analytical variable because valid price
information was available for only 9.25% of products.

Bundle-related language was retained as a diagnostic flag rather than an
exclusion criterion because preliminary filtering removed a substantial
share of otherwise valid Smart TV listings.

Reviews were restricted to 2015–2023. Empty review texts and exact
duplicate rows were removed, while short but meaningful reviews such as
"Great" or "Love it" were retained.

The resulting analytical dataset contains 1,097 Smart TV products and
111,191 consumer reviews.